In [0]:
%pip install -U "mlflow>=3.1.0" databricks-sdk
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 621.4/621.4 kB 23.6 MB/s eta 0:00:00
  Attempting uninstall: wcwidth
    Found existing installation: wcwidth 0.2.5
    Not uninstalling wcwidth at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-4ed1e5e6-84cc-4725-a024-a305f026a732
    Can't uninstall 'wcwidth'. No files were found to uninstall.
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside e

In [0]:
import mlflow
import os

mlflow.set_registry_uri("databricks-uc")

CATALOG="ai_ap_south_1"
SCHEMA="default"
MODEL_NAME = "gemma4_27b_vllm_wrapper"

SERVICE_NAME = "gemma4_27b_service"
model_code_path = "gemma4_27b_chat_model.py"

In [0]:
from gemma4_27b_chat_model import enable_openai_tool_compatibility

enable_openai_tool_compatibility()

In [ ]:
# def log_and_register_chat_model(
#     *,
#     model_code_path: str,
#     registered_model_name: str,
#     model_name: str = "model",
# ):
#     enable_openai_tool_compatibility()
#     previous_value = os.environ.get(LOGGING_MODE_ENV)
#     os.environ[LOGGING_MODE_ENV] = "1"
#     try:
#         return mlflow.pyfunc.log_model(
#             name=model_name,
#             python_model=model_code_path,
#             pip_requirements=PIP_REQUIREMENTS,
#             streamable=True,
#             registered_model_name=registered_model_name,
#         )
#     finally:
#         if previous_value is None:
#             os.environ.pop(LOGGING_MODE_ENV, None)
#         else:
#             os.environ[LOGGING_MODE_ENV] = previous_value

In [0]:
registered_model_name = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"

pip_requirements = [
    "mlflow>=3.1.0",
    "vllm==0.26.0",
    "transformers==5.14.1",
    "filelock==3.18.0",
    "httpx==0.28.1",
]


with mlflow.start_run():
    model_info = mlflow.pyfunc.log_model(
        name="model",
        python_model=model_code_path,
        pip_requirements=pip_requirements,
        # signature=MODEL_SIGNATURE,
        # input_example=None,
        streamable=True,
        registered_model_name=registered_model_name,
    )

model_version = str(model_info.registered_model_version)
print("Registered version:", model_version)


🔗 View Logged Model at: https://dbc-7fc2c08f-b1b1.cloud.databricks.com/ml/experiments/859606649907921/models/m-1a139ad8e90b4efe8c5491188e0138ec?o=7474660271867744
2026/08/19 17:29:52 WARNING mlflow.pyfunc: Default values for temperature, n and stream in ChatParams will be removed in the next release. Specify them in the input example explicitly if needed.
2026/08/19 17:29:52 INFO mlflow.pyfunc: Predicting on input example to validate output


2026-08-19 17:29:52,725 INFO Gemma4VLLMChatModel Skipping vLLM startup in a Databricks notebook


2026/08/19 17:29:54 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - vllm (current: uninstalled, required: vllm==0.26.0)
 - transformers (current: uninstalled, required: transformers==5.14.1)
 - filelock (current: 3.17.0, required: filelock==3.18.0)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2026/08/19 17:29:54 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - vllm (current: uninstalled, required: vllm==0.26.0)
 - transformers (current: uninstalled, required: transformers==5.14.1)
 - filelock (current: 3.17.0, required: filelock==3.18.0)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencie

2026-08-19 17:29:54,139 INFO Gemma4VLLMChatModel Skipping vLLM startup in a Databricks notebook


Registered model 'ai_ap_south_1.default.gemma4_27b_vllm_wrapper' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '5' of model 'ai_ap_south_1.default.gemma4_27b_vllm_wrapper': https://dbc-7fc2c08f-b1b1.cloud.databricks.com/explore/data/models/ai_ap_south_1/default/gemma4_27b_vllm_wrapper/version/5?o=7474660271867744


Registered version: 5


In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedModelInput,
    ServedModelInputWorkloadType,
)
from databricks.sdk.errors import ResourceDoesNotExist

ENDPOINT_NAME = "gemma4-27b-chat-vllm"
wc = WorkspaceClient()

endpoint_config = EndpointCoreConfigInput(
    name=ENDPOINT_NAME,
    served_models=[
        ServedModelInput(
            name=f"{MODEL_NAME}",
            model_name=registered_model_name,
            model_version=str(model_version),
            workload_size="Small",
            workload_type=ServedModelInputWorkloadType.MULTIGPU_MEDIUM,  # A10G
            scale_to_zero_enabled=True,
            environment_vars={
                "PYTHONUNBUFFERED": "1",
                "VLLM_USE_FLASHINFER_SAMPLER": "0",
                "GEMMA4_MAX_MODEL_LEN": "8192",
                "GEMMA4_GPU_MEMORY_UTILIZATION": "0.90",
                "GEMMA4_MAX_NUM_SEQS": "4",
            },
        )
    ]
)

# Check if endpoint already exists
try:
    existing_endpoint = wc.serving_endpoints.get(name=ENDPOINT_NAME)
    print(f"Updating existing endpoint: {ENDPOINT_NAME}")
    wc.serving_endpoints.update_config(
        name=ENDPOINT_NAME,
        served_models=endpoint_config.served_models,
    )
except ResourceDoesNotExist:
    print(f"Creating new endpoint: {ENDPOINT_NAME}")
    wc.serving_endpoints.create(
        name=ENDPOINT_NAME,
        config=endpoint_config,
    )

print(f"Endpoint '{ENDPOINT_NAME}' update submitted. It will be ready in sometime.")

Creating new endpoint: gemma4-27b-chat-vllm
Endpoint 'gemma4-27b-chat-vllm' update submitted. It will be ready in sometime.


Testing the endpoint

In [0]:
import mlflow.deployments

client = mlflow.deployments.get_deploy_client("databricks")

response = client.predict(
    endpoint="gemma4-27b-chat-vllm",
    inputs={
        "messages": [
            {
                "role": "user",
                "content": "What is the weather in Boston?",
            }
        ],
        "tools": [
            {
                "type": "function",
                "function": {
                    "name": "get_weather",
                    "description": "Get the current weather for a location",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "location": {
                                "type": "string",
                                "description": "City and state or country",
                            }
                        },
                        "required": ["location"],
                    },
                },
            }
        ],
        "custom_inputs": {
            "vllm": {
                "tool_choice": "auto",
            }
        },
        "temperature": 0.1,
        "max_tokens": 256,
    },
)


In [0]:
choice = response["choices"][0]
message = choice["message"]

if message.get("tool_calls"):
    for tool_call in message["tool_calls"]:
        function = tool_call["function"]

        print("Call ID:", tool_call["id"])
        print("Function:", function["name"])
        print("Arguments:", function["arguments"])

Call ID: chatcmpl-tool-b118aa1f5613a261
Function: get_weather
Arguments: {"location": "Boston"}


In [0]:
events = client.predict_stream(
    endpoint="gemma4-27b-chat-vllm",
    inputs={
        "messages": [
            {
                "role": "user",
                "content": "What is the weather in Boston?",
            }
        ],
        "tools": [
            {
                "type": "function",
                "function": {
                    "name": "get_weather",
                    "description": "Get weather for a location",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "location": {"type": "string"}
                        },
                        "required": ["location"],
                    },
                },
            }
        ],
        "custom_inputs": {
            "vllm": {
                "tool_choice": "auto",
            }
        },
    },
)

for event in events:
    for choice in event.get("choices", []):
        delta = choice.get("delta", {})

        if delta.get("content"):
            print(delta["content"], end="", flush=True)

        for tool_call in delta.get("tool_calls", []):
            print("\nTool call:", tool_call)



Tool call: {'function': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, 'id': 'chatcmpl-tool-adebd3ced0f3235f', 'type': 'function'}
